In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)


In [2]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [3]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    state['joke'] = response
    return state

In [4]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    state['explanation'] = response
    return state

In [5]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [6]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': '',
 'explanation': 'To explain a joke effectively, I need the specific joke you\'d like me to analyze! However, to demonstrate the process, here\'s an example using a classic pun:\n\n**Joke:**  \n*"Why don’t scientists trust atoms? Because they make up everything."*\n\n**Explanation:**  \n1. **Type of Humor:** This is a *pun*, relying on wordplay.  \n2. **Breakdown:**  \n   - **"Make up"** has a double meaning:  \n     - *Literal:* To constitute or compose (e.g., atoms are the building blocks of matter).  \n     - *Figurative:* To fabricate or lie (e.g., "He made up a story").  \n   - **"Trust"** implies reliability, but scientists distrust atoms because they’re the fundamental units of matter that *literally* make up all things, yet their behavior can be unpredictable at a quantum level.  \n3. **Punchline:** The humor arises from the unexpected twist of "make up" switching from a scientific fact to a playful accusation of dishonesty.  \n\nIf you provide yo

In [7]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '', 'explanation': 'To explain a joke effectively, I need the specific joke you\'d like me to analyze! However, to demonstrate the process, here\'s an example using a classic pun:\n\n**Joke:**  \n*"Why don’t scientists trust atoms? Because they make up everything."*\n\n**Explanation:**  \n1. **Type of Humor:** This is a *pun*, relying on wordplay.  \n2. **Breakdown:**  \n   - **"Make up"** has a double meaning:  \n     - *Literal:* To constitute or compose (e.g., atoms are the building blocks of matter).  \n     - *Figurative:* To fabricate or lie (e.g., "He made up a story").  \n   - **"Trust"** implies reliability, but scientists distrust atoms because they’re the fundamental units of matter that *literally* make up all things, yet their behavior can be unpredictable at a quantum level.  \n3. **Punchline:** The humor arises from the unexpected twist of "make up" switching from a scientific fact to a playful accusation of dishonesty.  \n

In [8]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '', 'explanation': 'To explain a joke effectively, I need the specific joke you\'d like me to analyze! However, to demonstrate the process, here\'s an example using a classic pun:\n\n**Joke:**  \n*"Why don’t scientists trust atoms? Because they make up everything."*\n\n**Explanation:**  \n1. **Type of Humor:** This is a *pun*, relying on wordplay.  \n2. **Breakdown:**  \n   - **"Make up"** has a double meaning:  \n     - *Literal:* To constitute or compose (e.g., atoms are the building blocks of matter).  \n     - *Figurative:* To fabricate or lie (e.g., "He made up a story").  \n   - **"Trust"** implies reliability, but scientists distrust atoms because they’re the fundamental units of matter that *literally* make up all things, yet their behavior can be unpredictable at a quantum level.  \n3. **Punchline:** The humor arises from the unexpected twist of "make up" switching from a scientific fact to a playful accusation of dishonesty.  \

In [9]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why don\'t skeletons eat pasta?  \nBecause they have *no guts* for it!  \n\n*(This joke plays on the double meaning of "guts" — both the physical organs skeletons lack and the courage needed to enjoy a hearty meal. It’s a classic pun with a spooky twist!)* 🍝👻',
 'explanation': 'The joke "Why don\'t skeletons eat pasta? Because they have *no guts* for it!" relies on a clever **double meaning** of the word "guts":  \n\n1. **Literal Meaning**: Skeletons, by definition, lack internal organs (like the stomach and intestines), so they physically can’t digest pasta.  \n2. **Figurative Meaning**: "Guts" is also slang for courage or determination ("I don’t have the guts to do that"). The punchline humorously twists the idea of needing courage to enjoy a meal into a literal anatomical impossibility.  \n\nThe humor arises from the **unexpected connection** between these two meanings, paired with the spooky imagery of skeletons. It’s a lighthearted **pun** that plays o

In [10]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why don\'t skeletons eat pasta?  \nBecause they have *no guts* for it!  \n\n*(This joke plays on the double meaning of "guts" — both the physical organs skeletons lack and the courage needed to enjoy a hearty meal. It’s a classic pun with a spooky twist!)* 🍝👻', 'explanation': 'The joke "Why don\'t skeletons eat pasta? Because they have *no guts* for it!" relies on a clever **double meaning** of the word "guts":  \n\n1. **Literal Meaning**: Skeletons, by definition, lack internal organs (like the stomach and intestines), so they physically can’t digest pasta.  \n2. **Figurative Meaning**: "Guts" is also slang for courage or determination ("I don’t have the guts to do that"). The punchline humorously twists the idea of needing courage to enjoy a meal into a literal anatomical impossibility.  \n\nThe humor arises from the **unexpected connection** between these two meanings, paired with the spooky imagery of skeletons. It’s a lighthearted *

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '', 'explanation': 'To explain a joke effectively, I need the specific joke you\'d like me to analyze! However, to demonstrate the process, here\'s an example using a classic pun:\n\n**Joke:**  \n*"Why don’t scientists trust atoms? Because they make up everything."*\n\n**Explanation:**  \n1. **Type of Humor:** This is a *pun*, relying on wordplay.  \n2. **Breakdown:**  \n   - **"Make up"** has a double meaning:  \n     - *Literal:* To constitute or compose (e.g., atoms are the building blocks of matter).  \n     - *Figurative:* To fabricate or lie (e.g., "He made up a story").  \n   - **"Trust"** implies reliability, but scientists distrust atoms because they’re the fundamental units of matter that *literally* make up all things, yet their behavior can be unpredictable at a quantum level.  \n3. **Punchline:** The humor arises from the unexpected twist of "make up" switching from a scientific fact to a playful accusation of dishonesty.  \n

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '', 'explanation': 'To explain a joke effectively, I need the specific joke you\'d like me to analyze! However, to demonstrate the process, here\'s an example using a classic pun:\n\n**Joke:**  \n*"Why don’t scientists trust atoms? Because they make up everything."*\n\n**Explanation:**  \n1. **Type of Humor:** This is a *pun*, relying on wordplay.  \n2. **Breakdown:**  \n   - **"Make up"** has a double meaning:  \n     - *Literal:* To constitute or compose (e.g., atoms are the building blocks of matter).  \n     - *Figurative:* To fabricate or lie (e.g., "He made up a story").  \n   - **"Trust"** implies reliability, but scientists distrust atoms because they’re the fundamental units of matter that *literally* make up all things, yet their behavior can be unpredictable at a quantum level.  \n3. **Punchline:** The humor arises from the unexpected twist of "make up" switching from a scientific fact to a playful accusation of dishonesty.  \

### Fault Tolerance

In [13]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [14]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    interrupted: int

In [15]:
# 2. Define steps
did_interrupt = False

def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    global did_interrupt
    print("⏳ Step 2 simulating interruption...")
    if not did_interrupt:
        did_interrupt = True
        raise KeyboardInterrupt("Simulated crash at Step 2")
    print("✅ Step 2 executed")
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [16]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [17]:
did_interrupt = False

try:
    print("▶️ Running graph: simulating interruption during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Simulated interruption captured.")

▶️ Running graph: simulating interruption during Step 2...
✅ Step 1 executed
⏳ Step 2 simulating interruption...
❌ Simulated interruption captured.


In [18]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 simulating interruption...
✅ Step 2 executed
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [19]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16561e-a734-674b-8003-5695f201a2fb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-06-11T06:51:04.485255+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16561e-a72f-6923-8002-089977326009'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16561e-a72f-6923-8002-089977326009'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-11T06:51:04.483254+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16561e-a033-6f88-8001-f4820bf96d29'}}, tasks=(PregelTask(id='3b2a301a-7f12-9354-f1e2-810d15fbb61